In [0]:
%python
dbutils.widgets.text("catalog","dab_proj_cat")
dbutils.widgets.text("schema","dab_proj_schema")
catalog=dbutils.widgets.get("catalog")
schema=dbutils.widgets.get("schema")

In [0]:
%python
query = f"""
CREATE OR REPLACE TEMPORARY VIEW temp_view1 AS
SELECT Id, AccountId, LastName, FirstName, Salutation, Name, Phone, Title
FROM {catalog}.{schema}.contact
"""
spark.sql(query)

In [0]:
%python
table_scd1 = f"`{catalog}`.`{schema}`.`contact_scd1`"
table_scd2 = f"`{catalog}`.`{schema}`.`contact_scd2`"
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {table_scd1} (
    Id STRING,
    AccountId STRING, 
    LastName STRING,
    FirstName STRING,
    Salutation STRING,
    Name STRING,
    Phone STRING,
    Title STRING,
    updated_at TIMESTAMP
)
USING DELTA
""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {table_scd2} (
    Id STRING,
    AccountId STRING, 
    LastName STRING,
    FirstName STRING,
    Salutation STRING,
    Name STRING,
    Phone STRING,
    Title STRING,
    start_date TIMESTAMP,
    end_date TIMESTAMP,
    is_active BOOLEAN
)
USING DELTA
""")

In [0]:
%python
target_table = f"`{catalog}`.`{schema}`.`contact_scd1`"
merge_query = f"""
MERGE INTO {target_table} AS tar
USING temp_view1 AS sou
ON tar.id = sou.id
WHEN MATCHED AND (
    tar.AccountId != sou.AccountId OR
    tar.LastName != sou.LastName OR
    tar.FirstName != sou.FirstName OR
    tar.Salutation != sou.Salutation OR
    tar.Name != sou.Name OR
    tar.Phone != sou.Phone OR
    tar.Title != sou.Title
)
THEN UPDATE SET
    tar.AccountId = sou.AccountId,
    tar.LastName = sou.LastName,
    tar.FirstName = sou.FirstName,
    tar.Salutation = sou.Salutation,
    tar.Name = sou.Name,
    tar.Phone = sou.Phone,
    tar.Title = sou.Title,
    tar.updated_at = current_timestamp()
WHEN NOT MATCHED BY TARGET
THEN INSERT (id, AccountId, LastName, FirstName, Salutation, Name, Phone, Title, updated_at)
VALUES (sou.id, sou.AccountId, sou.LastName, sou.FirstName, sou.Salutation, sou.Name, sou.Phone, sou.Title, current_timestamp())
WHEN NOT MATCHED BY SOURCE
THEN DELETE
"""

# 4️⃣ Execute merge
spark.sql(merge_query)

In [0]:
%python
scd2_table = f"`{catalog}`.`{schema}`.`contact_scd2`"
insert_query = f"""
INSERT INTO {scd2_table}
SELECT 
    s.Id,
    s.AccountId, 
    s.LastName,
    s.FirstName,
    s.Salutation,
    s.Name,
    s.Phone,
    s.Title,
    current_timestamp() AS start_date,
    NULL AS end_date,
    TRUE AS is_active
FROM temp_view1 s
JOIN {scd2_table} t
    ON s.id = t.id
WHERE t.is_active = TRUE
  AND (
      t.AccountId  != s.AccountId OR 
      t.LastName  != s.LastName OR 
      t.FirstName  != s.FirstName OR 
      t.Salutation  != s.Salutation OR 
      t.Name  != s.Name OR 
      t.Phone  != s.Phone OR 
      t.Title  != s.Title
  )
"""

spark.sql(insert_query)

# 4️⃣ Merge to expire old records and insert unmatched new ones
merge_query = f"""
MERGE INTO {scd2_table} AS tar
USING temp_view1 AS sou
ON tar.id = sou.id AND tar.is_active = TRUE
WHEN MATCHED AND (
      tar.AccountId  != sou.AccountId OR 
      tar.LastName  != sou.LastName OR 
      tar.FirstName  != sou.FirstName OR 
      tar.Salutation  != sou.Salutation OR 
      tar.Name  != sou.Name OR 
      tar.Phone  != sou.Phone OR 
      tar.Title  != sou.Title
)
THEN UPDATE SET tar.end_date = current_date(), tar.is_active = FALSE
WHEN NOT MATCHED BY TARGET
THEN INSERT (id, AccountId, LastName, FirstName, Salutation, Name, Phone, Title, start_date, end_date, is_active)
     VALUES (sou.id, sou.AccountId, sou.LastName, sou.FirstName, sou.Salutation, sou.Name, sou.Phone, sou.Title, current_timestamp(), NULL, TRUE)
WHEN NOT MATCHED BY SOURCE
THEN UPDATE SET tar.end_date = current_date(), tar.is_active = FALSE
"""

spark.sql(merge_query)